# Lesson 10: Overfitting, Regularization & Bias–Variance

*ML Academy — companion notebook*

Why models overfit and how L1/L2 penalties and the bias–variance trade-off help.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-10.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 11. In Python

In [1]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV
from sklearn.pipeline import make_pipeline

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def build(reg):
    return make_pipeline(PolynomialFeatures(2, include_bias=False), StandardScaler(), reg)

models = [("no regularization", LinearRegression())] + \
         [(f"Ridge alpha={a}", Ridge(alpha=a)) for a in [1, 10, 100]]
for name, reg in models:
    m = build(reg).fit(X_train, y_train)
    print(f"{name:20s} train R2 {m.score(X_train, y_train):.3f}  test R2 {m.score(X_test, y_test):.3f}")

for a in [0.1, 1, 5]:
    m = build(Lasso(alpha=a, max_iter=50000)).fit(X_train, y_train)
    coef = m[-1].coef_
    print(f"Lasso alpha={a}: test R2 {m.score(X_test, y_test):.3f}, non-zero weights {np.sum(coef != 0)} of {coef.size}")

m = build(RidgeCV(alphas=np.logspace(-2, 3, 30))).fit(X_train, y_train)
print("best alpha", round(m[-1].alpha_, 2), "test R2", round(m.score(X_test, y_test), 3))

no regularization    train R2 0.606  test R2 0.416
Ridge alpha=1        train R2 0.601  test R2 0.456
Ridge alpha=10       train R2 0.590  test R2 0.503
Ridge alpha=100      train R2 0.561  test R2 0.498
Lasso alpha=0.1: test R2 0.479, non-zero weights 54 of 65
Lasso alpha=1: test R2 0.516, non-zero weights 36 of 65
Lasso alpha=5: test R2 0.488, non-zero weights 12 of 65
best alpha 92.37 test R2 0.5


## 14. Going deeper: regularization in practice

In [2]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

# Measure bias² and variance directly: refit on 300 fresh training sets
rng = np.random.default_rng(0)
f = lambda x: np.sin(2 * np.pi * x)                   # the true curve
x_test = np.linspace(0, 1, 50)
for lam in [1e-6, 1e-3, 1e-1, 10]:
    preds = []
    for _ in range(300):
        x = rng.uniform(0, 1, 20); y = f(x) + rng.normal(0, 0.3, 20)
        m = make_pipeline(PolynomialFeatures(9), Ridge(alpha=lam)).fit(x[:, None], y)
        preds.append(m.predict(x_test[:, None]))
    preds = np.array(preds)
    bias2 = np.mean((preds.mean(axis=0) - f(x_test)) ** 2)
    var = np.mean(preds.var(axis=0))
    print(f"lambda {lam:7.0e}: bias² {bias2:.3f}   variance {var:.3f}   expected error {bias2 + var + 0.09:.3f}")

lambda   1e-06: bias² 0.001   variance 1.133   expected error 1.224


lambda   1e-03: bias² 0.028   variance 0.153   expected error 0.270


lambda   1e-01: bias² 0.100   variance 0.050   expected error 0.240


lambda   1e+01: bias² 0.335   variance 0.025   expected error 0.450


In [3]:
# Lasso's key step: soft-thresholding, used by coordinate descent
soft = lambda z, t: np.sign(z) * np.maximum(np.abs(z) - t, 0)
for z in [3.0, 0.8, -0.3, -2.5]:
    print(f"least-squares value {z:5.1f} -> lasso value {soft(z, 1.0) + 0.0:5.1f}    ridge value {z / (1 + 1.0):5.2f}")

least-squares value   3.0 -> lasso value   2.0    ridge value  1.50
least-squares value   0.8 -> lasso value   0.0    ridge value  0.40
least-squares value  -0.3 -> lasso value   0.0    ridge value -0.15
least-squares value  -2.5 -> lasso value  -1.5    ridge value -1.25


In [4]:
from sklearn.linear_model import LassoCV, ElasticNetCV
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler

X, y = load_diabetes(return_X_y=True)
X = StandardScaler().fit_transform(X)
lasso = LassoCV(cv=5, random_state=0).fit(X, y)
enet = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9], cv=5, random_state=0).fit(X, y)
print("lasso: best alpha", round(lasso.alpha_, 3), " non-zero weights", np.sum(lasso.coef_ != 0), "of 10")
print("elastic net: best alpha", round(enet.alpha_, 3), " l1_ratio", enet.l1_ratio_, " non-zero", np.sum(enet.coef_ != 0))

lasso: best alpha 0.079  non-zero weights 9 of 10
elastic net: best alpha 0.217  l1_ratio 0.9  non-zero 8


In [5]:
# Lasso with two almost identical features, on bootstrap samples: which one does it keep?
rng = np.random.default_rng(1)
from sklearn.linear_model import Lasso
n = 200
z = rng.normal(size=n)                                    # one underlying quantity…
a = z + rng.normal(0, 0.05, n); b = z + rng.normal(0, 0.05, n)   # …measured twice
yy = 3 * z + rng.normal(0, 1, n)
picks = {"only a": 0, "only b": 0, "both": 0}
for _ in range(200):
    i = rng.integers(0, n, n)
    w = Lasso(alpha=1.0).fit(np.column_stack([a, b])[i], yy[i]).coef_
    key = "both" if (w != 0).all() else ("only a" if w[0] != 0 else "only b")
    picks[key] += 1
print("lasso, 200 bootstrap samples:", picks)
from sklearn.linear_model import ElasticNet
w = ElasticNet(alpha=1.0, l1_ratio=0.5).fit(np.column_stack([a, b]), yy).coef_
print("elastic net weights on the full data:", w.round(2))

lasso, 200 bootstrap samples: {'only a': 47, 'only b': 63, 'both': 90}
elastic net weights on the full data: [0.89 0.89]


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [6]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
cols = ["size_m2", "bedrooms", "dist_km", "age_yrs"]
for name, reg in [("OLS", LinearRegression()), ("Ridge a=1", Ridge(alpha=1)), ("Ridge a=10", Ridge(alpha=10))]:
    m = make_pipeline(StandardScaler(), reg).fit(df[cols], df["price_lakh"])
    print(f"{name:11s}", m[-1].coef_.round(2))

OLS         [ 11.06   7.01 -14.42  -6.92]
Ridge a=1   [  7.72   8.32 -12.62  -5.6 ]
Ridge a=10  [ 3.79  4.93 -5.94 -2.63]


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [7]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Ridge by hand

One feature, no intercept: compute the ridge weight for λ = 5.

In [8]:
sum_xy, sum_xx, lam = 50.0, 20.0, 5.0
w = None  # ← your code here
check(w, 2.0, hint='w = Σxy / (Σx² + λ).')

❌ Not yet. Hint: w = Σxy / (Σx² + λ).


### Challenge 2: Lasso switches features off

Fit Lasso(alpha=1.0) and count how many weights are exactly zero.

In [9]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Lasso
X, y = load_diabetes(return_X_y=True)
model = Lasso(alpha=1.0).fit(X, y)
n_zero = None  # ← your code here
check(n_zero, 7, hint='Compare model.coef_ with 0 and count the True values.')

❌ Not yet. Hint: Compare model.coef_ with 0 and count the True values.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Ridge by hand** — the missing line:

```python
w = sum_xy / (sum_xx + lam)
```

**2. Lasso switches features off** — the missing line:

```python
n_zero = int(np.sum(model.coef_ == 0))
```